In [105]:
import os
import requests
import certifi
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_groq import ChatGroq
from langchain import hub

In [106]:
from langchain.agents import create_react_agent, AgentExecutor

In [107]:
os.environ["SSL_CERT_FILE"]= certifi.where()
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")


In [108]:
search_tool = TavilySearchResults(max_results=1)

result = search_tool.invoke("what is the latest news on AI")

result

[{'url': 'https://www.artificialintelligence-news.com',
  'content': 'Retail & Logistics AI\n\nJune 19, 2026\n\n### McDonald’s tests Google-backed AI drive-thru ordering system\n\nAI in Action\n\nJune 10, 2026\n\n#### Deep Learning\n\n### Aviva deploys AI to stop £230M in sophisticated insurance fraud\n\nAI in Action\n\nJune 8, 2026\n\n### China’s AI just mapped its entire renewable energy grid. Here’s why the rest of the world should pay attention\n\nEnvironment & Sustainability\n\nMay 22, 2026\n\n### IBM Research unveils breakthrough analog AI chip for efficient deep learning\n\nAI Market Trends\n\nAugust 11, 2023\n\n## Subscribe\n\nAll our premium content and latest tech news delivered straight to your inbox\n\n## Resources\n\nAI and Us, Interviews\n\nApril 21, 2026\n\n# Uniquely Wired podcast with Microsoft’s Agnes Heftberger\n\nResource\n\nJanuary 8, 2026 [...] HSBC expands AI banking partnership with Google Cloud\n\nAI in Action\n\nJune 18, 2026\n\n# HSBC expands AI banking partn

In [ ]:
@tool

def get_weather_data(city: str)-> str :
    """
    fetch current weather information for a city.

    """
    url=(
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"could not fetch weather data for  {city}"

    return (
        f"city = {city}\n"
        f"temperature: {data['current']['temperature']}\n"
        f"weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}\n"
    )

    

In [110]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    # model = "llama-3.1-8b-instant",
    temperature =0,
    api_key = GROQ_API_KEY
)

In [111]:
response = llm.invoke("TELL ME A JOKE ABOUT AI")
response

AIMessage(content='Why did the AI program go on a diet?\n\nBecause it wanted to lose some bytes! (get it?)', response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 43, 'total_tokens': 66, 'completion_time': 0.061250587, 'completion_tokens_details': None, 'prompt_time': 0.001930303, 'prompt_tokens_details': None, 'queue_time': 0.161845465, 'total_time': 0.06318089}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_3272ea2d91', 'finish_reason': 'stop', 'logprobs': None}, id='run-62e89824-dd74-4a09-9fbc-8db9d2c79660-0')

In [112]:
from groq import Groq
from dotenv import load_dotenv
import os

load_dotenv()

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

models = client.models.list()

for model in models.data:
    print(model.id)

openai/gpt-oss-120b
whisper-large-v3-turbo
llama-3.1-8b-instant
openai/gpt-oss-safeguard-20b
canopylabs/orpheus-v1-english
whisper-large-v3
meta-llama/llama-4-scout-17b-16e-instruct
openai/gpt-oss-20b
meta-llama/llama-prompt-guard-2-22m
llama-3.3-70b-versatile
qwen/qwen3.6-27b
allam-2-7b
groq/compound
canopylabs/orpheus-arabic-saudi
groq/compound-mini
meta-llama/llama-prompt-guard-2-86m
qwen/qwen3-32b


In [113]:
prompt = hub.pull("hwchase17/react")

In [114]:
tools=[search_tool, get_weather_data]

agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt= prompt
)

In [115]:
agent_executor = AgentExecutor(
    agent=agent,
    tools= tools,
    verbose = True,
     max_iterations=3,
     handle_parsing_errors=True
)
import warnings

warnings.filterwarnings("ignore")

In [ ]:
response = agent_executor.invoke(
    {
        "input" :(
            "find the capital of france and its current humidity"
            # "and then find its current weather"
        )
    }
)
print(response["output"])